# 05 — The estimator and its parameters

*Chapter 08 · Gradient Boosting · Notebook 5 of 6*

Notebooks 1–4 built the gradient-boosting machine by hand and learned its three dials (ν, depth, the number
of trees). Now we use the real scikit-learn estimator, `GradientBoostingRegressor` /
`GradientBoostingClassifier`, and meet the parameters that matter in practice — gathered around one spine:
**early stopping**, the principled cure for the overfitting Notebook 4 made you watch.

**Prerequisites.** Notebooks 1–4 (the residual loop, the gradient view, classification with the Newton
leaf, and ν / depth / n_estimators); Chapter 06 Notebook 4 and Chapter 07 Notebook 4 (reading an
estimator's parameters honestly); Chapter 06 Notebook 5 (permutation importance).

**What you'll be able to do.**
- Drive `GradientBoosting*` and choose its `loss`.
- Let **early stopping** pick the number of trees automatically.
- Regularize with `subsample` (stochastic gradient boosting).
- Read MDI against permutation importance.
- Tune honestly, and know when to reach for `HistGradientBoosting*`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_friedman1
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, train_test_split

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()
SEED = 0

# The same Friedman benchmark as Notebook 4: 5 informative features (including the
# x0*x1 interaction) + 5 pure-noise features, so we know the ground truth.
X_raw, y = make_friedman1(n_samples=2000, noise=1.0, random_state=SEED)
X = pd.DataFrame(X_raw, columns=[f"x{i}" for i in range(X_raw.shape[1])])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=SEED)
print(f"train {X_train.shape[0]} rows, test {X_test.shape[0]};  "
      f"{X.shape[1]} features (5 informative + 5 noise)")

## A recap, and the question

Notebook 4 left us with a practical question: how many trees, and at what learning rate? Before answering
it, one reassurance — **the estimator is exactly the machine we built**. By hand, our residual loop matched
`GradientBoostingRegressor` to about 1e-16 for regression (Notebook 1), and `GradientBoostingClassifier` to
about 3e-15 for classification once we used the Newton leaf (Notebook 3). So from here we can turn the
library's knobs knowing precisely what each one does to the loop underneath.

We work with `GradientBoostingRegressor` (and its twin `GradientBoostingClassifier`), on the same Friedman
data as Notebook 4 — five informative features, five noise, a known structure we can hold the model
accountable to.

## The dials, and the `loss`

The most important choice is the **`loss`** — Notebook 2 showed that the loss *is* the method. For
regression: `'squared_error'` (the default), `'absolute_error'`, `'huber'` (robust), `'quantile'`
(prediction intervals). For classification: `'log_loss'` (the default) and `'exponential'` — which, as
Notebook 3 showed, is AdaBoost's objective. (The older names `'deviance'`, `'ls'`, `'lad'` have been
**removed** from current scikit-learn.)

The remaining dials we already know: `learning_rate` and `n_estimators` (the ν × trees trade-off,
Notebook 4), `max_depth` (interaction order, kept shallow — Notebook 4), and `subsample` (below).

**One API trap.** Unlike `AdaBoostClassifier`, the gradient-boosting estimators have **no `staged_score`** —
only `staged_predict` (and `staged_decision_function` / `staged_predict_proba`). To watch a metric tree by
tree, compute it yourself over `staged_predict`, exactly as we do below.

## Early stopping: let the data choose the number of trees

Notebook 4's lesson was that too many trees overfit at a large ν, and that even at a small ν you still have
to pick `n_estimators`. Rather than guess, **early stopping** holds out a small **validation slice** of the
training data (`validation_fraction`), tracks the model's score on it after every tree, and **stops once
that score stops improving** for `n_iter_no_change` rounds (by more than `tol`). You ask for a large number
of trees; the data tells you when to quit.

In [ ]:
# Request a large forest; let a held-out validation slice decide when to stop.
full = GradientBoostingRegressor(
    n_estimators=2000, learning_rate=0.1, max_depth=3, random_state=SEED
).fit(X_train, y_train)
stopped = GradientBoostingRegressor(
    n_estimators=2000, learning_rate=0.1, max_depth=3,
    validation_fraction=0.1, n_iter_no_change=10, tol=1e-4, random_state=SEED,
).fit(X_train, y_train)

# Staged test R² of the full model, tree by tree (GB has no staged_score).
test_r2 = np.array([r2_score(y_test, p) for p in full.staged_predict(X_test)])
n_stop = stopped.n_estimators_
print(f"requested 2000 trees  ->  early stopping used {n_stop}")
print(f"test R²:  full 2000-tree {full.score(X_test, y_test):.4f}   "
      f"early-stopped {stopped.score(X_test, y_test):.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(1, full.n_estimators_ + 1), test_r2, color=COLORS["model"], linewidth=2,
        label="full model: test R²")
ax.axvline(n_stop, color=COLORS["highlight"], linestyle="--", linewidth=1.8,
           label=f"early stop: {n_stop} trees")
ax.set_xscale("log")
ax.set_xlabel("number of trees")
ax.set_ylabel("test R²")
ax.set_title("early stopping halts where the held-out score plateaus")
ax.legend()
plt.show()

**Read the figure.** The full model's test R² climbs to about 0.93 by roughly 150 trees, then runs
**flat** all the way to 2000. Early stopping (the dashed line) halts at **142** trees — and its test R²
(0.930) is actually a touch *above* the full 2000-tree model (0.927), which has begun the gentle ν=0.1
overfit we saw in Notebook 4. So early stopping saved about 1860 trees at no cost in accuracy, with no
guessing. The exact stop point drifts with the seed (it tracks a small held-out validation slice, so a
different split stops at a somewhat different tree), though the test R² stays close to 0.93 either way —
that sensitivity is the honest cost of letting the data decide.

## `subsample`: stochastic gradient boosting

By default each tree is fit on all the training rows. Set `subsample < 1` and each tree sees a fresh random
**fraction** of the rows (Friedman 2002). This is the bagging idea from Chapter 06 — row randomness —
dropped inside a booster: it **decorrelates** the successive steps (a regularizer) and makes each tree
**cheaper** to fit. A useful by-product: with `subsample < 1` the estimator also exposes `oob_improvement_`,
an out-of-bag running estimate of progress.

In [ ]:
# subsample < 1 fits each tree on a random fraction of the rows (Friedman 2002).
fractions = [0.25, 0.5, 0.75, 1.0]
train_r2, test_r2_ss = [], []
for ss in fractions:
    m = GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.1, max_depth=3, subsample=ss, random_state=SEED
    ).fit(X_train, y_train)
    train_r2.append(m.score(X_train, y_train))
    test_r2_ss.append(m.score(X_test, y_test))
    print(f"subsample={ss}: test R²={test_r2_ss[-1]:.4f}  train R²={train_r2[-1]:.4f}")

fig = viz.plot_train_test_curve(
    fractions, train_r2, test_r2_ss, xlabel="subsample fraction", ylabel="R²"
)
ax = fig.axes[0]
ax.set_xticks(fractions)
ax.set_title("stochastic gradient boosting: sampling rows regularizes")
plt.show()

**Read the figure.** Sampling **half to three-quarters** of the rows per tree (subsample 0.5–0.75)
reaches a *higher* test R² than using all of them (≈ 0.936 vs 0.929) — the same variance-cutting effect
bagging gave a forest (Chapter 06), now helping a booster, and at lower cost per tree. Two caveats, kept
honest: too aggressive (subsample 0.25) hands the gain back, and the free `oob_improvement_` estimate is
**noisier** than a held-out validation set (Friedman 2002), so for choosing the number of trees, prefer the
validation-based early stopping above.

## Which features did the model use?

A fitted gradient-boosting model exposes `feature_importances_` — the **MDI** (mean impurity reduction) we
met for trees and forests (Chapters 04, 06). Because MDI has known biases (it favours high-cardinality /
continuous features), we cross-check it with **permutation importance** (Chapter 06 Notebook 5): shuffle one
feature in the test set and measure how much the score drops. Friedman's data is the perfect honesty test —
we *know* x₀–x₄ carry the signal (with x₀ and x₁ interacting) and x₅–x₉ are pure noise.

In [ ]:
gbr = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.1, max_depth=3, random_state=SEED
).fit(X_train, y_train)
mdi = gbr.feature_importances_
perm = permutation_importance(gbr, X_test, y_test, n_repeats=10, random_state=SEED)
print(f"MDI sum:  informative x0-x4 = {mdi[:5].sum():.3f}   noise x5-x9 = {mdi[5:].sum():.3f}")

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.5))
viz.plot_feature_importances(X.columns, mdi, top=10, ax=axL, label="MDI")
axL.set_title("MDI (impurity reduction)")
viz.plot_feature_importances(X.columns, perm.importances_mean, std=perm.importances_std,
                             top=10, ax=axR, color=COLORS["highlight"], label="permutation")
axR.set_title("permutation importance (test set)")
plt.show()

**Read the figure.** Both panels tell the same story: the model leans on the **five informative**
features (their MDI sums to 0.988) and assigns essentially **zero** to the **five noise** features. MDI and
permutation **agree on the ranking of the informative features** — x₃ largest (the strongest single-feature
linear term, while the larger x₀·x₁ interaction signal is split between its two features), then the x₀ / x₁
pair, then x₄ and x₂. MDI's biases are mild here because the
noise features are genuinely uninformative, and permutation confirms the picture on held-out data. A
reminder from Chapter 06: importance is about **use, not cause** — it says what the model relied on, not
what drives the world.

## Tuning, honestly

The dials interact, so search them **together** with cross-validation on the **training** set, then report
a single **sealed test** score — never the cross-validation winner as your headline number (the lesson of
Chapters 06 and 07). Here is a small grid over `learning_rate` and `max_depth`.

In [ ]:
grid = GridSearchCV(
    GradientBoostingRegressor(n_estimators=300, random_state=SEED),
    {"learning_rate": [0.05, 0.1, 0.2], "max_depth": [2, 3]},
    cv=5, scoring="r2", n_jobs=-1,
).fit(X_train, y_train)
default = GradientBoostingRegressor(n_estimators=300, random_state=SEED).fit(X_train, y_train)

print(f"best params: {grid.best_params_}   (best CV R² {grid.best_score_:.4f})")
print(f"sealed-test R²:  tuned {grid.best_estimator_.score(X_test, y_test):.4f}   "
      f"default {default.score(X_test, y_test):.4f}")

**Read the result.** The grid's best combination is `{learning_rate: 0.1, max_depth: 3}` — which
**is** scikit-learn's default. On the sealed test the tuned model scores 0.929, exactly matching the
default's 0.929: tuning bought **nothing** here. That is a real and common outcome — the defaults (ν = 0.1,
depth 3) are well-chosen for many tabular problems. Tune when you have a reason to, always validate on data
the search never saw, and judge the model by that sealed number.

## The fast modern default: `HistGradientBoosting*`

Everything so far used the classic `GradientBoosting*`, where each of `n_estimators` is one tree we could
rebuild by hand. For larger datasets scikit-learn offers a faster pair, `HistGradientBoostingRegressor` /
`HistGradientBoostingClassifier`, which:

- bin each feature into a **histogram** of at most `max_bins = 255` values (so split-finding scans bins, not
  sorted rows),
- grow trees **leaf-wise** to `max_leaf_nodes = 31` rather than by a fixed depth,
- turn **`early_stopping='auto'`** on by default,
- and handle **categorical features and missing values natively**.

It is usually much faster, and similar or better in accuracy. It is also the bridge to the next two
chapters: **XGBoost** (Chapter 09) and **LightGBM** (Chapter 10) refine exactly this histogram, leaf-wise,
regularized skeleton. We put `HistGradientBoosting*` head-to-head with the classic estimator on real data in
the capstone, Notebook 6.

## Your turn

1. **Why it stops.** From the early-stopping figure, explain in one sentence why stopping at ~142 trees
   costs nothing, even though we asked for 2000.
2. **Move the stop.** Re-fit the early-stopped model with `n_iter_no_change=5` (more impatient) or
   `tol=1e-3` (a coarser "no improvement" bar). Report how the stop point moves and whether the test R²
   changes.
3. **Why sampling helps.** `subsample=0.5` beat full-sample here. Explain why fitting each tree on half the
   rows can *raise* the test R² (relate it to bagging's variance cut in Chapter 06), and why that does not
   contradict boosting being a sequential method.

## What you built

- You drove the real `GradientBoosting*` estimator and chose its `loss` — confirming it is the by-hand
  machine of Notebooks 1–3.
- You used **early stopping** to pick the number of trees automatically (2000 requested → 142 used, for
  equal-or-better test R²).
- You regularized with **`subsample`** (stochastic gradient boosting) and saw row-sampling beat full-sample.
- You read **MDI against permutation importance** on data with a known structure, and watched both recover
  the signal and ignore the noise.
- You tuned honestly to one sealed test, and met **`HistGradientBoosting*`**, the fast modern default.

**Vocabulary you now own:** `n_iter_no_change` / `validation_fraction` / early stopping · `subsample` /
stochastic gradient boosting / `oob_improvement_` · MDI vs permutation importance · `staged_predict` (and
the missing `staged_score`) · `HistGradientBoosting*`.

## Going further (optional)

- **Early stopping** spends a `validation_fraction` of the training data and depends on `tol` and
  `n_iter_no_change`; it is **off by default** (`n_iter_no_change=None`). For small datasets a held-out
  slice is noisy — cross-validated stopping or `HistGradientBoosting*`'s built-in version can be steadier.
- **`subsample`** is Friedman's *stochastic gradient boosting* (2002). The companion `oob_improvement_` is
  convenient but noisy; a proper validation set is more reliable.
- **`loss='huber'`** down-weights large residuals (robust regression); **`loss='quantile'`** with `alpha`
  predicts a quantile, giving prediction intervals.
- The fast path — histograms, leaf-wise growth, an explicitly regularized objective — is the subject of
  Chapters 09 (XGBoost) and 10 (LightGBM).

## References

- Friedman, J. H. (2001). *Greedy function approximation: a gradient boosting machine.* Annals of
  Statistics, 29(5), 1189–1232. DOI [10.1214/aos/1013203451](https://doi.org/10.1214/aos/1013203451).
- Friedman, J. H. (2002). *Stochastic gradient boosting.* Computational Statistics & Data Analysis, 38(4),
  367–378. DOI [10.1016/S0167-9473(01)00065-2](https://doi.org/10.1016/S0167-9473(01)00065-2).
  (`subsample`, out-of-bag improvement.)
- Friedman, J. H. (1991). *Multivariate adaptive regression splines.* Annals of Statistics, 19(1), 1–67.
  DOI [10.1214/aos/1176347963](https://doi.org/10.1214/aos/1176347963). (The `make_friedman1` function.)
- Breiman, L. (2001). *Random forests.* Machine Learning, 45(1), 5–32. DOI
  [10.1023/A:1010933404324](https://doi.org/10.1023/A:1010933404324). (Permutation importance.)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §10.11–10.13. DOI [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).

*Previous: 04 — Shrinkage and the trees.*  ·  *Next: 06 — A demanding case: California housing.*